# Lab 8 — Let an AI attack your project
### EP0418 · Topic 9 · 1 practical hour

---
Today you point an AI assistant at your own pipeline and ask it to **find what is
wrong with it**.

> **What is assessed is your rebuttal, not what the critic finds.**

The critic will raise things that are wrong. Saying *why* they are wrong, with
evidence, is worth exactly as much as fixing the ones that are right.

> **First: File → Save a copy in Drive, and work in your copy.** This page opened from GitHub, and changes here are not saved. Reopen your copy from Google Drive next time.

## Set up Gemini for this notebook (once, 30 seconds)

Open Gemini (the spark icon at the bottom of the notebook). In its chat box, click the **sliders icon** next to **+**, choose **Configure**, paste the text below and save. Leave the mode on **Default**. Colab keeps the instructions with this notebook, so every answer follows the course rules.

```
I am new to coding. Explain any code you write in plain words: what it does to my data.
Before writing code for a bigger task, tell me your plan in plain words.
Never delete or drop rows unless I ask. If a step would remove rows, say so first.
After changing the data, show the row count before and after, and the smallest and largest values.
Codes such as -999 mean a missing reading. Point them out; do not treat them as real numbers.
Do not change any line that starts with check. Those are the lab's locked checks.
If you are not sure, say so. Do not guess.
```

In [ ]:
import os, urllib.request
REPO = "https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/"


def fetch(*names):
    """Download any course file that is not beside the notebook yet."""
    for name in names:
        if os.path.exists(name):
            continue
        try:
            urllib.request.urlretrieve(REPO + name, name)
            print("fetched", name)
        except Exception:
            print(f"Could not download {name}. Upload it with the folder icon on the left.")
            from google.colab import files
            files.upload()


fetch("ep0418.py")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from ep0418 import check


def filled(*values):
    """True once every TODO slot above has been replaced. Otherwise says so."""
    if any(v is ... or v is None for v in values):
        print("A TODO above is still '...'. Fill it in, then run this cell again.")
        return False
    return True

# Run this first. The pre-lab cells need these files beside the notebook.
fetch("smart_building_sensors.csv", "smart_self_daily.csv", "smart_money_ohlcv.csv")

## Pre-lab A — Three checks your labs never used

Labs 0–7 all ran on classroom sensor data and a transport feed. Two important
checks never came up, because that data does not contain the problems they catch —
but **your track might**. Run this cell at home before the lab and read what happens.

In [ ]:
# --- zeros_are_real: is a zero a measurement, or a device that was off? -------
# In the building data, occupancy = 0 means the room was empty. A REAL zero.
b = pd.read_csv("smart_building_sensors.csv", parse_dates=["timestamp"])
check.reset()
check.zeros_are_real(b, "occupancy", max_zeros=20000)      # passes — real zeros

# In personal activity data, steps = 0 usually means the watch was in a drawer.
# Same value, completely different meaning.
if os.path.exists("smart_self_daily.csv"):
    s = pd.read_csv("smart_self_daily.csv", parse_dates=["date"])
    check.zeros_are_real(s, "steps")                       # NOTE — investigate

# --- no_missing: are there blanks left in what you are about to analyse? ------
check.no_missing(b, cols=["co2_ppm", "temp_c"])

check.report()

### Read those two `zeros_are_real` results

`occupancy = 0` **passed**. `steps = 0` came back as a **NOTE**.

The check cannot tell you which is which — you told it, by setting `max_zeros`.
An empty room really does have zero people in it. A person with a watch in a
drawer did not really take zero steps, and averaging those days in drags every
statistic you compute downward.

**If your track has a column where zero might mean "not recorded", decide now**
and write the decision into your CA3 report.

## Pre-lab B — Lookahead bias (Track 1, and anyone acting on a signal)

If you compute a signal from today's data and then act on today's data, you used
information you would not have had at the time. This is the single most common way
a student backtest produces an impressive and completely fake result.

In [ ]:
# TODO(you): only if your project acts on a signal. Otherwise read and move on.
if os.path.exists("smart_money_ohlcv.csv"):
    m = pd.read_csv("smart_money_ohlcv.csv", parse_dates=["Date"])
    raw_signal = (m.Close > m.Close.rolling(20).mean()).astype(int)

    check.reset()
    check.signal_is_shifted(raw_signal, raw_signal, by=1)        # FAILS - lookahead
    check.report()
    check.signal_is_shifted(raw_signal, raw_signal.shift(1), by=1)  # PASSES
    check.report()

The first call fails and the second passes, on the same data. The only difference
is one `.shift(1)` — acting on yesterday's signal instead of today's.

If your system triggers an action from a signal, **prove the shift is there.**

## Task 1 — Run the critic

Open a **new chat** for this, not the one that wrote your code: a fresh chat has
not seen your reasoning and has no work of its own to defend. Copy the prompt below
into ChatGPT, Claude, Copilot or Cursor, then paste in your own pipeline code: the
five stage functions and `pipeline()` from Lab 7. (In Colab, click in a cell,
Ctrl+A, Ctrl+C.)

---
```
You are reviewing a student data pipeline for a module on automated decision
systems. Be adversarial but specific — your job is to find real defects, not to
be encouraging. Flag only defects that affect correctness or the checks below;
do not suggest style changes or extra features.

Check for each of these and say PASS, FAIL or UNSURE with a one-line reason:

1. Silent row loss — does any step drop rows without accounting for them?
2. Sentinel values — are non-measurement codes (-999, 0, 9999) handled?
3. Zero vs missing — are zeros that mean "no reading" distinguished from real zeros?
4. Time index — is the data resampled to a regular grid before any .shift()?
5. Horizon mismatch — are the model and baseline scored at the same horizon?
6. Lookahead bias — does any signal use information from the future?
7. Baseline honesty — is the baseline the strongest one a sensible person would use?
8. Response layer — is the alert debounced and rate-limited?
9. Failure mode — what happens if the data source returns nothing?

For each FAIL give the specific line and what you would change.

Here is my pipeline:
[PASTE YOUR CODE]
```
---

## Task 2 — Record every finding, then answer it

Fill this in. **One row per finding.** Your verdict column is what gets marked.

| # | Probe | Critic's finding | My verdict | Evidence |
|---|---|---|---|---|
| 1 | Silent row loss | | Accept / Reject / Unsure | |
| 2 | Sentinel values | | Accept / Reject / Unsure | |
| 3 | Zero vs missing | | Accept / Reject / Unsure | |
| 4 | Time index | | Accept / Reject / Unsure | |
| 5 | Horizon mismatch | | Accept / Reject / Unsure | |
| 6 | Lookahead bias | | Accept / Reject / Unsure | |
| 7 | Baseline honesty | | Accept / Reject / Unsure | |
| 8 | Response layer | | Accept / Reject / Unsure | |
| 9 | Failure mode | | Accept / Reject / Unsure | |

**Accept** = it is right, and here is what I changed.
**Reject** = it is wrong, and here is the evidence that it is wrong.
**Unsure** = I could not tell; here is what I would need to check. A PASS from
the critic still gets a row: write "critic said PASS" and whether you agree.

A row that says "Reject — I checked rows 400-460 and the resample keeps the empty
slots as NaN, it does not drop them" is worth more than one that says "Accept".

## Task 3 — Fix what deserves fixing, then re-verify

For each finding you accepted, ask your assistant to fix it **and to test its fix**:

```
Fix this: [the finding]. Then write a test that shows the fix works:
print the row count before and after, and the smallest and largest
value of the columns it touches. Do not change any existing check.
```

Then judge its test, not only its answer. For each fix, write down:

- **What did the test actually check?** "It returns a table" proves nothing.
- **Did it change or remove any `check.` line?** It may not.
- **Did it show numbers, or only say "passed"?** A pass with no numbers is a claim, not evidence.

In [ ]:
import pandas as pd, numpy as np
import os
from ep0418 import check

# TODO(you): re-run your pipeline's checks after your fixes
...
check.report()

## The question the critic cannot answer

An AI critic can check your code. It cannot tell you **what happens on the days
your system is wrong, and who pays for it.** That question is the Week 14 record
box on your CA3 Build 3 sheet, and it goes straight into your CA3 reflection.
Think about it this week; write it next week.

---
## Read before you run

A Track 5 group asked the critic to check their weekly activity alert. The critic said PASS on “zero versus missing”. The code was:

**Do not run it yet.** Read it, answer the two questions in the cell below,
and only then remove the `#` and run it to see whether you were right.

Was the critic right? Eleven days in this file have `steps` equal to 0 because the watch was not worn. What do those days do to `weekly`, and to `df.steps.mean()`?

Write down how many days the rule flags with the zeros left in, and how many after the zeros are replaced with NaN (run `zeros_are_real` and then try both).

In [ ]:
# --- AI-generated. Read it first. ---
# df = pd.read_csv("smart_self_daily.csv", parse_dates=["date"])
# weekly = df.steps.rolling(7).mean()
# low_activity = weekly < 0.7 * df.steps.mean()

**What is wrong with it:**

**What it would produce, and what it should produce:**

---
## Before you leave

- The findings table is complete, with a verdict and evidence on **every** row.
- `verification_log.md` is up to date across all eight labs.

**Reality check:** if you accepted every finding without pushing back on any, you
probably have not read them closely enough. Critics are confidently wrong too —
that is the whole lesson of this module.